# 🧠 Binary Classification: Predicting Age Group from LLM Interactions
### Machine Learning Project – Google Colab Notebook

**Goal:** Build the best binary classification model to predict whether a user belongs to the *Young Adults* or *Older Adults* group, based on their interactions with a Large Language Model (ChatGPT).

**Dataset:** `all_tasks_90_sub_23_12.csv` – 1,275 conversation records between users and GPT, with metadata and demographic labels.

**Target Variable:** `subject_group` (Young_Adults vs Older_Adults)

---
**Table of Contents:**
1. Environment Setup & Imports
2. Data Loading & Exploratory Data Analysis
3. Data Cleaning & Preprocessing
4. Feature Engineering
5. Feature Selection
6. Train/Test Split & Scaling
7. Model Training & Comparison
8. Hyperparameter Tuning
9. Final Evaluation
10. Summary & Conclusions

## 1. Environment Setup & Imports

### 1.1 Install Required Libraries
We install all necessary packages that are not pre-installed on Google Colab.

In [ ]:
!pip install textstat xgboost nltk scikit-learn transformers torch

import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
print("✅ All packages installed and NLTK data downloaded.")

### 1.2 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix, roc_curve, auc)
from xgboost import XGBClassifier
from scipy.stats import loguniform, randint, uniform

# NLP
import textstat
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
import string
from collections import Counter

# Plotting style
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully.")

## 2. Data Loading & Exploratory Data Analysis (EDA)

### 2.1 Load the Dataset
Upload the CSV file to Colab's `/content/` directory, then load it into a pandas DataFrame.

In [ ]:
# Load the dataset
df = pd.read_csv('/content/all_tasks_90_sub_23_12.csv')
df_original = df.copy()  # Keep an unmodified copy for reference

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn names:\n{list(df.columns)}")
df.head()

### 2.2 Data Overview & Statistics

In [ ]:
# Data types and non-null counts
print("="*60)
print("DATA TYPES & NULL COUNTS")
print("="*60)
print(df.dtypes)
print(f"\nTotal missing values per column:")
print(df.isnull().sum())
print(f"\nBasic statistics for numerical columns:")
df.describe()

### 2.3 Target Variable Distribution

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
target_counts = df['subject_group'].value_counts()
colors = ['#2ecc71', '#e74c3c']
target_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Subject Group')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.03, 0.03), shadow=True,
            textprops={'fontsize': 12})
axes[1].set_title('Target Variable Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nClass balance ratio: {target_counts.min()/target_counts.max():.2f} — fairly balanced ✅")

### 2.4 Feature Overview

In [ ]:
# Quick look at unique values per column
print("UNIQUE VALUES PER COLUMN:")
print("="*50)
for col in df.columns:
    n_unique = df[col].nunique()
    dtype = df[col].dtype
    sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else 'N/A'
    sample_str = str(sample)[:50] + '...' if len(str(sample)) > 50 else str(sample)
    print(f"  {col:45s} | {str(dtype):10s} | {n_unique:5d} unique | e.g. {sample_str}")

# Numerical features distribution
num_cols = ['msg_count_within_p', 'Age', 'response_time_sec', 'response_time_min', 'words_in_message_to_gpt']
fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 4))
for i, col in enumerate(num_cols):
    df.boxplot(column=col, by='subject_group', ax=axes[i])
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
plt.suptitle('Numerical Features by Subject Group', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Preprocessing

### 3.1 Drop ID & Timestamp Columns
These columns are identifiers with no predictive value. We also encode the target variable.

> **Important Note on `Age`:** The `Age` feature has ~0.97 correlation with the target variable `subject_group`. This is expected since the groups *are* defined by age ranges, making `Age` essentially a label proxy. **We exclude `Age` from all models** to ensure the model learns meaningful behavioral patterns, not just age thresholds.

In [ ]:
# Columns to drop (IDs, timestamps, raw text handled separately, and Age)
columns_to_drop = [
    'msg_id', 'gpt_interface_id', 'participant_id', 'subject_id',
    'unix_time_when_msg_sent_to_gpt', 'date_time_when_msg_sent_to_gpt',
    'unix_time_when_msg_received_from_gpt', 'date_time_when_msg_received_from_gpt',
]

# Encode target variable
df['target'] = df['subject_group'].map({'Young_Adults': 0, 'Older_Adults': 1})
print(f"Target encoding: Young_Adults → 0, Older_Adults → 1")
print(f"Target distribution:\n{df['target'].value_counts()}")

# Keep only raw text columns for DistilBERT (no raw/basic engineered features)
text_to_gpt = df['message_to_gpt'].astype(str).fillna('')
text_from_gpt = df['message_from_gpt'].astype(str).fillna('')
combined_text = (text_to_gpt + ' [SEP] ' + text_from_gpt).astype(str)

# Rebuild modeling dataframe to prevent leakage from non-text raw columns
df = pd.DataFrame({'target': df['target'].astype(int)})

print('Using only DistilBERT input text (message_to_gpt + [SEP] + message_from_gpt).')
print(f"Rows: {len(df)}")

## 4. Feature Engineering

### 4.1 DistilBERT Contextual Embeddings
We use DistilBERT to extract a single 768-dimensional embedding from combined text (`message_to_gpt [SEP] message_from_gpt`).

> ⚠️ This cell uses GPU acceleration. Make sure to enable GPU in Colab: *Runtime → Change runtime type → GPU*

### 4.1 DistilBERT Contextual Embeddings
We use **DistilBERT** (a lighter, faster version of BERT) to extract deep contextual embeddings from the text.
These embeddings capture semantic meaning that TF-IDF and BoW cannot, encoding relationships between words in context.
We keep the full 768-dimensional `[CLS]` embedding from the combined text (`message_to_gpt [SEP] message_from_gpt`).

> ⚠️ This cell uses GPU acceleration. Make sure to enable GPU in Colab: *Runtime → Change runtime type → GPU*

In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel

# Load DistilBERT model and tokenizer
print("Loading DistilBERT model...")
tokenizer_bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_bert = model_bert.to(device)
model_bert.eval()
print(f"✅ DistilBERT loaded on {device}")

def get_bert_embedding(text, max_length=128):
    text = str(text)[:512]  # Truncate very long texts
    encoded = tokenizer_bert(text, return_tensors='pt', padding='max_length',
                             truncation=True, max_length=max_length)
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        output = model_bert(**encoded)
    # Use [CLS] token embedding (first token)
    cls_embedding = output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return cls_embedding

# Extract embeddings from combined text
print("Extracting DistilBERT embeddings from combined text...")
bert_embeddings = []
for i, text in enumerate(combined_text):
    bert_embeddings.append(get_bert_embedding(text))
    if (i + 1) % 200 == 0:
        print(f"  Processed {i+1}/{len(combined_text)} messages...")
bert_array = np.array(bert_embeddings)

bert_df = pd.DataFrame(bert_array, columns=[f'bert_{i}' for i in range(bert_array.shape[1])], index=df.index)

# Add only 768 BERT features to main dataframe
df = pd.concat([df, bert_df], axis=1)
print(f"\n✅ Added {bert_array.shape[1]} DistilBERT embedding features. DataFrame shape: {df.shape}")

### 4.2 Assemble Final Feature Matrix

In [ ]:
# Separate target from features
y = df['target']
X = df.drop(columns=['target'])

# Fill any remaining NaN values
X = X.fillna(0)

# Ensure all numeric
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)
X = X.astype(np.float64)

bert_cols = [c for c in X.columns if c.startswith('bert_')]
if len(bert_cols) != 768:
    raise ValueError(f'Expected exactly 768 DistilBERT features, found {len(bert_cols)}')
if len(X.columns) != 768:
    raise ValueError(f'Expected feature matrix to have exactly 768 columns, found {len(X.columns)}')

print(f"Final feature matrix: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Target distribution: {dict(y.value_counts())}")
print(f"DistilBERT features: {len(bert_cols)}")

## 6. Train/Test Split & Scaling

In [ ]:
# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set:     {X_test_scaled.shape[0]} samples")
print(f"Features:     {X_train_scaled.shape[1]}")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

## 7. Model Training & Comparison

### 7.1 Train Multiple Models (including ANN)
We train several classifiers — including a Keras ANN — and compare their performance using 3-fold cross-validation on the training set, then evaluate on the held-out test set.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Define sklearn models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=2000, solver='liblinear'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', n_estimators=200),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'LinearSVC': LinearSVC(random_state=42, dual=False, max_iter=5000),
    'Naive Bayes': GaussianNB(),
}

# Train and evaluate sklearn models
results = []
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    results.append({'Model': name, 'CV F1 (mean)': cv_scores.mean(), 'CV F1 (std)': cv_scores.std(),
                    'Test Accuracy': acc, 'Test F1': f1, 'Test Precision': prec, 'Test Recall': rec})
    print(f"  {name:25s} | CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Test F1: {f1:.4f} | Test Acc: {acc:.4f}")

# --- ANN Model ---
print("\n  Training ANN (Keras)...")
input_dim = X_train_scaled.shape[1]

def build_ann():
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

ann_model = build_ann()
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
ann_model.fit(X_train_scaled, y_train, epochs=40, batch_size=32,
              validation_split=0.2, callbacks=[early_stop], verbose=0)

y_pred_ann = (ann_model.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
acc_ann = accuracy_score(y_test, y_pred_ann)
f1_ann = f1_score(y_test, y_pred_ann)
prec_ann = precision_score(y_test, y_pred_ann)
rec_ann = recall_score(y_test, y_pred_ann)
results.append({'Model': 'ANN (Keras)', 'CV F1 (mean)': np.nan, 'CV F1 (std)': np.nan,
                'Test Accuracy': acc_ann, 'Test F1': f1_ann, 'Test Precision': prec_ann, 'Test Recall': rec_ann})
print(f"  {'ANN (Keras)':25s} | Test F1: {f1_ann:.4f} | Test Acc: {acc_ann:.4f}")

results_df = pd.DataFrame(results).sort_values('Test F1', ascending=False).reset_index(drop=True)
print("\n" + "="*80)
print("MODEL COMPARISON (sorted by Test F1)")
print("="*80)
results_df

### 7.2 Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 Score comparison
results_sorted = results_df.sort_values('Test F1', ascending=True)
colors_f1 = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(results_sorted)))
axes[0].barh(results_sorted['Model'], results_sorted['Test F1'], color=colors_f1, edgecolor='black', alpha=0.85)
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Test F1 Score by Model', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1)
for i, v in enumerate(results_sorted['Test F1']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

# Accuracy comparison
axes[1].barh(results_sorted['Model'], results_sorted['Test Accuracy'], color=colors_f1, edgecolor='black', alpha=0.85)
axes[1].set_xlabel('Accuracy')
axes[1].set_title('Test Accuracy by Model', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 1)
for i, v in enumerate(results_sorted['Test Accuracy']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 7.3 End-to-End DistilBERT Fine-Tuning (GPU)
This step upgrades DistilBERT from a static feature extractor to a fully trainable classifier.

- Uses `DistilBertForSequenceClassification` with a trainable classification head
- Trains with **AdamW** (`lr=2e-5`, `weight_decay=0.01`)
- Uses weighted **CrossEntropyLoss** to mitigate class imbalance
- Runs on GPU when available (`device='cuda'`)
- Clears GPU cache at the end with `torch.cuda.empty_cache()`

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Fine-tuning device: {device}")

# Fine-tuning hyperparameters
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01

# Build a single text field from both conversation directions
combined_text = (text_to_gpt.astype(str) + " [SEP] " + text_from_gpt.astype(str))

# Reuse the same train/test split indices from Section 6
train_idx = X_train.index
test_idx = X_test.index

train_texts = combined_text.loc[train_idx].tolist()
test_texts = combined_text.loc[test_idx].tolist()
y_train_ft = y.loc[train_idx].astype(int).values
y_test_ft = y.loc[test_idx].astype(int).values

tokenizer_ft = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_ft = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
).to(device)

class ConversationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = ConversationDataset(train_texts, y_train_ft, tokenizer_ft, MAX_LEN)
test_ds = ConversationDataset(test_texts, y_test_ft, tokenizer_ft, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Class-weighted cross-entropy for imbalance handling
class_counts = np.bincount(y_train_ft, minlength=2)
class_weights = len(y_train_ft) / (2.0 * np.maximum(class_counts, 1))
class_weights_t = torch.tensor(class_weights, dtype=torch.float, device=device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_t)

optimizer = AdamW(model_ft.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(f"Class counts: {class_counts.tolist()} | class weights: {[round(w, 4) for w in class_weights.tolist()]}")
print(f"Training steps per epoch: {len(train_loader)}")

# Training loop
for epoch in range(EPOCHS):
    model_ft.train()
    running_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model_ft(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / max(len(train_loader), 1)
    print(f"Epoch {epoch+1}/{EPOCHS} - train loss: {avg_loss:.4f}")

# Evaluation
model_ft.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model_ft(input_ids=input_ids, attention_mask=attention_mask).logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

ft_acc = accuracy_score(all_true, all_preds)
ft_f1 = f1_score(all_true, all_preds)
ft_prec = precision_score(all_true, all_preds)
ft_rec = recall_score(all_true, all_preds)

ft_metrics = {
    'Model': 'DistilBERT Fine-Tuned',
    'Test Accuracy': ft_acc,
    'Test F1': ft_f1,
    'Test Precision': ft_prec,
    'Test Recall': ft_rec
}

print("\n✅ DistilBERT Fine-Tuning Results")
print(f"  Accuracy:  {ft_acc:.4f}")
print(f"  F1 Score:  {ft_f1:.4f}")
print(f"  Precision: {ft_prec:.4f}")
print(f"  Recall:    {ft_rec:.4f}")

print("\nClassification report (fine-tuned DistilBERT):")
print(classification_report(all_true, all_preds, target_names=['Young Adults', 'Older Adults']))

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 Cleared GPU cache with torch.cuda.empty_cache()")

## 8. Hyperparameter Tuning

### 8.1 Tune Top Models
We tune XGBoost, Gradient Boosting, Logistic Regression, and the ANN with compact search spaces for faster execution (~2-4 min total).

In [ ]:
tuned_results = []

# --- XGBoost Tuning (fast: 10 iter × 2cv) ---
print("--- Tuning XGBoost ---")
param_dist_xgb = {
    'n_estimators': randint(100, 400),
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth': randint(3, 8),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 5),
    'reg_alpha': loguniform(0.01, 5),
    'reg_lambda': loguniform(0.01, 5),
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=param_dist_xgb,
    n_iter=10, scoring='f1', cv=2, random_state=42, n_jobs=-1, verbose=0
)
xgb_search.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {xgb_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_xgb):.4f} | Test Acc: {accuracy_score(y_test, y_pred_xgb):.4f}")
tuned_results.append({'Model': 'XGBoost (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_xgb),
    'Test F1': f1_score(y_test, y_pred_xgb), 'Test Precision': precision_score(y_test, y_pred_xgb),
    'Test Recall': recall_score(y_test, y_pred_xgb), 'estimator': xgb_search.best_estimator_})

# --- Gradient Boosting Tuning (fast: 8 iter × 2cv) ---
print("\n--- Tuning Gradient Boosting ---")
param_dist_gb = {
    'n_estimators': randint(80, 300),
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth': randint(2, 6),
    'subsample': uniform(0.6, 0.4),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 6),
}
gb_search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist_gb,
    n_iter=8, scoring='f1', cv=2, random_state=42, n_jobs=-1, verbose=0
)
gb_search.fit(X_train_scaled, y_train)
y_pred_gb = gb_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {gb_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_gb):.4f} | Test Acc: {accuracy_score(y_test, y_pred_gb):.4f}")
tuned_results.append({'Model': 'Gradient Boosting (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_gb),
    'Test F1': f1_score(y_test, y_pred_gb), 'Test Precision': precision_score(y_test, y_pred_gb),
    'Test Recall': recall_score(y_test, y_pred_gb), 'estimator': gb_search.best_estimator_})

# --- Logistic Regression Tuning (fast: 8 iter × 2cv) ---
print("\n--- Tuning Logistic Regression ---")
param_dist_lr = [
    {'C': loguniform(0.001, 100), 'solver': ['liblinear'], 'penalty': ['l1', 'l2']},
    {'C': loguniform(0.001, 100), 'solver': ['lbfgs'], 'penalty': ['l2']},
]
lr_search = RandomizedSearchCV(
    LogisticRegression(random_state=42, max_iter=3000),
    param_distributions=param_dist_lr,
    n_iter=8, scoring='f1', cv=2, random_state=42, n_jobs=-1, verbose=0
)
lr_search.fit(X_train_scaled, y_train)
y_pred_lr = lr_search.best_estimator_.predict(X_test_scaled)
print(f"  Best CV F1: {lr_search.best_score_:.4f} | Test F1: {f1_score(y_test, y_pred_lr):.4f}")
tuned_results.append({'Model': 'Logistic Regression (tuned)', 'Test Accuracy': accuracy_score(y_test, y_pred_lr),
    'Test F1': f1_score(y_test, y_pred_lr), 'Test Precision': precision_score(y_test, y_pred_lr),
    'Test Recall': recall_score(y_test, y_pred_lr), 'estimator': lr_search.best_estimator_})

# --- ANN Tuning (manual grid: test different architectures) ---
print("\n--- Tuning ANN (Keras) ---")
best_ann_f1 = 0
best_ann_model = None

ann_configs = [
    {'layers': [128, 64, 32], 'dropout': 0.3, 'lr': 0.001},
    {'layers': [64, 32], 'dropout': 0.2, 'lr': 0.001},
]

for i, cfg in enumerate(ann_configs):
    tf.random.set_seed(42)
    m = Sequential([Input(shape=(input_dim,))])
    for units in cfg['layers']:
        m.add(Dense(units, activation='relu'))
        m.add(BatchNormalization())
        m.add(Dropout(cfg['dropout']))
    m.add(Dense(1, activation='sigmoid'))
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']),
              loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(X_train_scaled, y_train, epochs=40, batch_size=32,
          validation_split=0.2, callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)], verbose=0)
    y_p = (m.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
    f1_val = f1_score(y_test, y_p)
    print(f"  Config {i+1} {cfg['layers']} dr={cfg['dropout']} lr={cfg['lr']} → F1: {f1_val:.4f}")
    if f1_val > best_ann_f1:
        best_ann_f1 = f1_val
        best_ann_model = m
        best_ann_pred = y_p

tuned_results.append({'Model': 'ANN (tuned)', 'Test Accuracy': accuracy_score(y_test, best_ann_pred),
    'Test F1': f1_score(y_test, best_ann_pred), 'Test Precision': precision_score(y_test, best_ann_pred),
    'Test Recall': recall_score(y_test, best_ann_pred), 'estimator': best_ann_model})
print(f"  Best ANN F1: {best_ann_f1:.4f}")

tuned_df = pd.DataFrame(tuned_results).sort_values('Test F1', ascending=False).reset_index(drop=True)
print("\n" + "="*80)
print("TUNED MODEL COMPARISON")
print("="*80)
tuned_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']]

## 9. Final Evaluation

### 9.1 Select Best Model & Classification Report

In [ ]:
# Select the best overall model
best_row = tuned_df.iloc[0]
best_model = best_row['estimator']
best_model_name = best_row['Model']
y_pred_best = best_model.predict(X_test_scaled)

print(f"🏆 Best Model: {best_model_name}")
print(f"\n{'='*60}")
print("CLASSIFICATION REPORT")
print('='*60)
print(classification_report(y_test, y_pred_best, target_names=['Young Adults', 'Older Adults']))

### 9.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Young Adults', 'Older Adults'],
            yticklabels=['Young Adults', 'Older Adults'])
axes[0].set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Normalized Confusion Matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=['Young Adults', 'Older Adults'],
            yticklabels=['Young Adults', 'Older Adults'])
axes[1].set_title(f'Normalized Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

### 9.3 Feature Importance (if tree-based model)

In [ ]:
# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X_train_scaled.columns).sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = 20
    top_imp = importances.head(top_n).sort_values()
    colors_imp = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_imp)))
    top_imp.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='black')
    ax.set_title(f'Top {top_n} Feature Importances — {best_model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 10 most important features:")
    for i, (feat, imp) in enumerate(importances.head(10).items()):
        print(f"  {i+1:2d}. {feat:45s} | {imp:.4f}")
elif hasattr(best_model, 'coef_'):
    importances = pd.Series(np.abs(best_model.coef_[0]), index=X_train_scaled.columns).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 8))
    top_imp = importances.head(20).sort_values()
    top_imp.plot(kind='barh', ax=ax, color=plt.cm.viridis(np.linspace(0.3, 0.9, 20)), edgecolor='black')
    ax.set_title(f'Top 20 Feature Coefficients (abs) — {best_model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('|Coefficient|')
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model type.")

## 10. Summary & Conclusions

In [ ]:
# Final comprehensive comparison table
print("="*80)
print("FINAL MODEL COMPARISON TABLE")
print("="*80)

# Combine base + tuned results
all_results = pd.concat([results_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']],
                          tuned_df[['Model', 'Test Accuracy', 'Test F1', 'Test Precision', 'Test Recall']]])
all_results = all_results.sort_values('Test F1', ascending=False).reset_index(drop=True)
print(all_results.to_string())

print(f"\n{'='*80}")
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy:  {best_row['Test Accuracy']:.4f}")
print(f"   F1 Score:  {best_row['Test F1']:.4f}")
print(f"   Precision: {best_row['Test Precision']:.4f}")
print(f"   Recall:    {best_row['Test Recall']:.4f}")
print(f"{'='*80}")

### Key Findings & Discussion

**Dataset Context:**
- The dataset contains user-LLM conversation records with demographic labels
- **Target**: `subject_group` – Young Adults vs Older Adults
- The `Age` feature is excluded from modeling

**Feature Engineering (DistilBERT-only):**
- Model input uses only raw text: `message_to_gpt [SEP] message_from_gpt`
- DistilBERT `[CLS]` embedding produces exactly **768** numeric features
- No additional handcrafted/basic/raw-tabular features are used
- No feature-selection stage is applied